# Using MetaARIMA via StatsForecast

This notebook shows how to use MetaARIMA using StatsForecast API. Includes a comparison is the same — **MetaARIMA**, **AutoARIMA**, and **SeasonalNaive** on M3 Monthly.

Outline:
1. Load a pre-trained MetaARIMA model
2. Load the M3 Monthly dataset
3. Forecast with `MetaARIMAModel` inside `StatsForecast`
4. Evaluate and compare using MASE

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
# !pip install metaforecast catboost datasetsforecast

## 1. Load a pre-trained MetaARIMA

We load the monthly model trained on M4 Monthly (see notebook `08-metaarima` for how to train one from scratch). Since M3 Monthly is a different corpus, this tests out-of-sample generalisation of the meta-learner.

The object below is the meta-learner. `MetaARIMAModel` in the next section applies it to each series.

In [2]:
from metaforecast.coseal import MetaARIMA

meta_arima = MetaARIMA.load(freq="ME")

print(f"Model loaded: freq={meta_arima.freq}, season_length={meta_arima.season_length}")
print(f"Meta-learner fitted: {meta_arima.is_fit}")
print(f"Shortlist size (n_trials): {meta_arima.n_trials}")
print(f"Configurations in search space: {len(meta_arima.model_names)}")

Model loaded: freq=ME, season_length=12
Meta-learner fitted: True
Shortlist size (n_trials): 25
Configurations in search space: 400


## 2. Load M3 Monthly dataset

We use M3 Monthly from the `datasetsforecast` package and split each series into train/test with a horizon of 12 (standard M3 Monthly horizon).

In [3]:
import pandas as pd
from datasetsforecast.m3 import M3

from metaforecast.evaluation.cv import SeriesWiseSplit

HORIZON = 12
SEASON_LENGTH = 12
FREQ = "ME"
N_SERIES = 10

df, *_ = M3.load(".", group="Monthly")

train, test = SeriesWiseSplit.time_wise_split(df, horizon=HORIZON)

print(f"Total series: {df['unique_id'].nunique()}")
print(f"Train shape:  {train.shape}")
print(f"Test shape:   {test.shape}")

2026-10-12 20:06:22,767	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-10-12 20:06:23,291	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Total series: 1428
Train shape:  (150426, 3)
Test shape:   (17136, 3)


## 3. Forecast with StatsForecast

`MetaARIMAModel` takes the loaded meta-learner. StatsForecast calls it with each series' values and the horizon; the wrapper rebuilds a one-series frame, shortlists configurations, and selects by AICc.

`n_jobs=1` keeps the run sequential. Each series still gets its own selected ARIMA: `fit` stores that model on the per-series copy, not on the shared meta-learner.

In [4]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, SeasonalNaive

from metaforecast.coseal import MetaARIMAModel

sf_models = [
    MetaARIMAModel(meta_arima),
    AutoARIMA(season_length=SEASON_LENGTH),
    SeasonalNaive(season_length=SEASON_LENGTH),
]

MODEL_NAMES = ["MetaARIMA", "AutoARIMA", "SeasonalNaive"]

In [5]:
uids = train["unique_id"].unique().tolist()[:N_SERIES]
test_sub = test[test["unique_id"].isin(uids)]
bad = set(test_sub.loc[test_sub["y"].isna(), "unique_id"])
uids = [uid for uid in uids if uid not in bad]

train_sub = train[train["unique_id"].isin(uids)].reset_index(drop=True)
test_sub = test_sub[test_sub["unique_id"].isin(uids)].reset_index(drop=True)

sf = StatsForecast(models=sf_models, freq=FREQ, n_jobs=1)
sf.fit(train_sub)
fcst = sf.predict(h=HORIZON)

results_df = test_sub[["unique_id", "ds", "y"]].merge(fcst, on=["unique_id", "ds"])

In [6]:
results_df

,unique_id,ds,y,MetaARIMA,AutoARIMA,SeasonalNaive
0,M1,1994-09-30,1560.000000,2156.321594,3456.428571,4800.000000
1,M1,1994-10-31,1440.000000,2005.595445,3456.428571,3000.000000
2,M1,1994-11-30,240.000000,2005.595445,3456.428571,3120.000000
3,M1,1994-12-31,1800.000000,2005.595445,3456.428571,5880.000000
4,M1,1995-01-31,4680.000000,2005.595445,3456.428571,2640.000000
...,...,...,...,...,...,...
115,M1006,1993-10-31,6282.200195,6241.416072,6314.928600,6352.500000
116,M1006,1993-11-30,6271.750000,6236.873076,6308.697069,6324.000000
117,M1006,1993-12-31,6260.299805,6209.569575,6302.749190,6287.149902
118,M1006,1994-01-31,6279.100098,6204.175875,6301.125236,6269.299805


## 4. Evaluate with MASE

MASE (Mean Absolute Scaled Error) is the standard metric for M3. A value < 1 means the model beats the naive seasonal benchmark.

In [11]:
from utilsforecast.losses import mase

errors = mase(
    df=results_df,
    models=MODEL_NAMES,
    seasonality=SEASON_LENGTH,
    train_df=train,
)

errors

,unique_id,MetaARIMA,AutoARIMA,SeasonalNaive
0,M1,0.379488,0.786344,0.912260
1,M10,0.563062,0.557562,0.720375
2,M100,0.594926,0.502128,0.581064
3,M1000,0.276405,0.194587,0.721053
4,M1001,0.437691,0.719845,0.662985
5,M1002,0.381891,0.411611,1.737571
6,M1003,0.636021,0.725778,0.568637
7,M1004,0.209364,0.164930,0.254877
8,M1005,0.653037,0.736792,1.319021
9,M1006,0.176245,0.131401,0.295694
